In [0]:
#simple json
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

cust_schema = StructType([
    StructField("name", StringType()),
    StructField("age", IntegerType()),
    StructField("city", StringType())
])

df = spark.createDataFrame([
    ('{"name":"Vignesh","age":28,"city":"Chennai"}',)
], ["json_str"])

df.show()
df2=df.withColumn("json_data",from_json(col("json_str"),schema=cust_schema))
df2.show(truncate=False)

df2.select("json_data.name","json_data.age","json_data.city").show()

In [0]:
#array
df = spark.createDataFrame([
    (1, ["apple", "banana", "grape"]),
    (2, ["mango", "orange"])
], ["id", "fruits"])

from pyspark.sql.functions import explode,size,array_contains
#Explode array into multiple rows
df.select(col("id"),explode(col("fruits"))).show()
#Get element by index
df.select(col("id"),(col("fruits")[0])).show()
#Array size
df.select(size((col("fruits")))).show()
#Array contains
df.filter(array_contains(col("fruits"),"banana")).show(truncate=False)

In [0]:
#complex json
data = [
    ('{"user":{"name":"Vignesh","address":{"street":"MG Road","city":"Bangalore"},"phones":[{"type":"home","number":"123"},{"type":"work","number":"456"}]}}',)
]

df = spark.createDataFrame(data, ["json_str"])


df.show(truncate=False)

from pyspark.sql.types import *
schema = StructType([
    StructField("user", StructType([
        StructField("name", StringType()),
        StructField("address", StructType([
            StructField("street", StringType()),
            StructField("city", StringType())
        ])),
        StructField("phones", ArrayType(StructType([
            StructField("type", StringType()),
            StructField("number", StringType())
        ])))]))])

df_parsed=df.withColumn("payload",from_json(col("json_str"),schema))

df_parsed.show(truncate=False)

# Select nested fields and explode phones
df_parsed.select(
    col("payload.user.name").alias("name"),
    col("payload.user.address.city").alias("city"),
    explode(col("payload.user.phones")).alias("phone")
).select(
    "name", "city", col("phone.type").alias("phone_type"), col("phone.number").alias("phone_number")
).show(truncate=False)
                    

